In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
import pandas as pd
import numpy as np
import os

from astropy.time import Time

In [ ]:
# set the directory path (Epoch1 for RACSMid)
directory_path = 'epoch_5/'

# create an empty list to store dataframes
dfs = []

# loop through all files in the directory that start with "beam_inf"
for filename in os.listdir(directory_path):
    if filename.startswith('beam_inf') and filename.endswith('.csv'):
        # read the CSV file into a pandas dataframe
        filepath = os.path.join(directory_path, filename)
        df = pd.read_csv(filepath)
        # append the dataframe to the list
        dfs.append(df)
        
# concatenate all dataframes into a single dataframe
combined_df = pd.concat(dfs, ignore_index=True)


In [ ]:
# plot the beam time of each dataframe
plt.figure(figsize=(20, 5))
plt.scatter(combined_df['BEAM_TIME'], combined_df['BEAM_NUM'])
plt.title('Beam Time')
plt.xlabel('Time')
plt.ylabel('Beam Number')
plt.show()

## RACSMid1 Fields

In [ ]:
# RACSMid1 fields (Epoch 1): 1617 fields in total
df_field_data = pd.read_csv('epoch_5/field_data.csv')
# print(df_field_data['SCAN_START'])

In [ ]:
# remove all the invalid entries from the dataframe: 1492 fields left
# df_field_data = df_field_data[df_field_data['SCAN_START'] != -1]
# df_field_data = df_field_data[df_field_data['SELECT'] != 0]
# df_field_data = df_field_data[df_field_data['SCAN_LEN'] > 800]


# plot the scan start time against the scan time length for each scan
plt.figure(figsize=(20, 5))
plt.scatter(df_field_data['SCAN_START'], df_field_data['SCAN_LEN'])
plt.title('Scan Time')
plt.xlabel('Scan Start Time')
plt.ylabel('Scan Time Length')
plt.show()

In [ ]:
len(df_field_data)

In [ ]:
# plot the field vs the scan observation time
plt.figure(figsize=(10, 8))
plt.scatter(df_field_data['RA_DEG'], df_field_data['DEC_DEG'], c=df_field_data['SCAN_LEN'], cmap='jet')
plt.title('Field vs Observation Time')
plt.xlabel('RA (in degrees)')
plt.ylabel('DEC (in degrees)')
clb = plt.colorbar()
clb.set_label('Scan Length (in seconds)')
plt.show()


## MJD to UTC Function

In [ ]:
def mjd2utc(mjd_seconds):
    # create a Time object with the MJD seconds
    t = Time(mjd_seconds/86400, format='mjd', scale='utc')

    # convert the time to YYYYMMDD HH:MM:SS format
    time_str = t.datetime.strftime('%Y-%m-%d %H:%M:%S')
    t_frac = str(t).split('.')[1]
    fin_time = time_str + '.' + t_frac  
    return fin_time


## FIRST Field

In [ ]:
# create a new dataframe with fields that already overlap with those mapped by FIRST
df_req_fields1 = df_field_data[((df_field_data['RA_DEG'].between(136, 240)) & (df_field_data['DEC_DEG'].between(-10, 48)))]
df_req_fields2 = df_field_data[(((df_field_data['RA_DEG'] < 45) | (df_field_data['RA_DEG'] > 315)) & (df_field_data['DEC_DEG'].between(-10, 10)))]
df_req_fields_first = pd.concat([df_req_fields1, df_req_fields2], ignore_index=True)

In [ ]:
# plot the field vs the scan observation time
plt.figure(figsize=(10, 8))
plt.scatter(df_req_fields_first['RA_DEG'], df_req_fields_first['DEC_DEG'], c=df_req_fields_first['SCAN_LEN'], cmap='jet')
plt.title('Field vs Observation Time')
plt.xlabel('RA (in degrees)')
plt.ylabel('DEC (in degrees)')
plt.ylim(-40, 50)
clb = plt.colorbar()
clb.set_label('Scan Length (in seconds)')
plt.show()

In [ ]:
# save the field names to a numpy array for use in the crossmatch notebook
field_list = df_req_fields_first['FIELD_NAME'].tolist()
# np.save('RACSLow_Fields.npy', field_list)

sbid_list = df_req_fields_first['SBID'].tolist()
# np.save('RACSLow_SBIDs.npy', sbid_list)

cal_sbid_list = df_req_fields_first['CAL_SBID'].tolist()
# np.save('RACSLow_CAL_SBIDs.npy', cal_sbid_list)

df_req_fields_first['UTC_SCAN_START'] = df_req_fields_first['SCAN_START'].apply(mjd2utc)
time_list = df_req_fields_first['UTC_SCAN_START'].tolist()
# np.save('RACSLow_Times.npy', time_list)

# print(field_list)

# Create a new dataframe
df_list_first = pd.DataFrame()

# Assign values to the columns
df_list_first['Field Name'] = field_list
df_list_first['SBID'] = sbid_list
df_list_first['CAL_SBID'] = cal_sbid_list
df_list_first['UTC Scan Start Time'] = time_list

df_list_first.sort_values('UTC Scan Start Time', inplace=True)
df_list_first.drop_duplicates(subset='Field Name', keep='last', inplace=True)

In [ ]:
# Save the field names to a numpy array and then to a numpy file
np.save('RACSHigh1_FIRST.npy', np.array(df_list_first))

## VLASS Field

In [ ]:
# create a new dataframe with fields that already overlap with those mapped by VLASS
df_req_fields_vlass = df_field_data[(df_field_data['DEC_DEG'].between(-40, 50))]

In [ ]:
# create a new dataframe with fields that already overlap with those mapped by VLASS but excluding fields above DEC +30 deg and galactic plane
df_req_fields_vlass_dec = df_req_fields_vlass[df_req_fields_vlass['DEC_DEG'] < 30]
df_req_fields_vlass_gal = df_req_fields_vlass_dec[(df_req_fields_vlass_dec['GAL_LAT'] > 11) | (df_req_fields_vlass_dec['GAL_LAT'] < -11)]

In [ ]:
# create a new dataframe with fields that already overlap with those mapped outside VLASS
df_req_fields_out_vlass = df_field_data[(df_field_data['DEC_DEG'].between(-90, -40))]

In [ ]:
# plot the field vs the scan observation time
plt.figure(figsize=(10, 8))
plt.scatter(df_req_fields_out_vlass['RA_DEG'], df_req_fields_out_vlass['DEC_DEG'], c=df_req_fields_out_vlass['SCAN_LEN'], cmap='jet')
plt.title('Field vs Observation Time')
plt.xlabel('RA (in degrees)')
plt.ylabel('DEC (in degrees)')
plt.ylim(-90, 50)
clb = plt.colorbar()
clb.set_label('Scan Length (in seconds)')
plt.show()

In [ ]:
# plot the field vs the scan observation time
plt.figure(figsize=(10, 8))
plt.scatter(df_req_fields_vlass_gal['RA_DEG'], df_req_fields_vlass_gal['DEC_DEG'], c=df_req_fields_vlass_gal['SCAN_LEN'], cmap='jet')
plt.title('Field vs Observation Time')
plt.xlabel('RA (in degrees)')
plt.ylabel('DEC (in degrees)')
plt.ylim(-90, 50)
clb = plt.colorbar()
clb.set_label('Scan Length (in seconds)')
plt.show()

In [ ]:
# save the field names to a numpy array for use in the crossmatch notebook
field_list = df_req_fields_out_vlass['FIELD_NAME'].tolist()
# np.save('RACSLow_Fields.npy', field_list)

sbid_list = df_req_fields_out_vlass['SBID'].tolist()
# np.save('RACSLow_SBIDs.npy', sbid_list)

cal_sbid_list = df_req_fields_out_vlass['CAL_SBID'].tolist()
# np.save('RACSLow_CAL_SBIDs.npy', cal_sbid_list)

df_req_fields_out_vlass['UTC_SCAN_START'] = df_req_fields_out_vlass['SCAN_START'].apply(mjd2utc)
time_list = df_req_fields_out_vlass['UTC_SCAN_START'].tolist()
# np.save('RACSLow_Times.npy', time_list)

# print(field_list)

# Create a new dataframe
df_list_vlass = pd.DataFrame()

# Assign values to the columns
df_list_vlass['Field Name'] = field_list
df_list_vlass['SBID'] = sbid_list
df_list_vlass['CAL_SBID'] = cal_sbid_list
df_list_vlass['UTC Scan Start Time'] = time_list

df_list_vlass.sort_values('UTC Scan Start Time', inplace=True)
df_list_vlass.drop_duplicates(subset='Field Name', keep='last', inplace=True)

In [ ]:
# Save the field names to a numpy array and then to a numpy file
np.save('RACSLow3_outsideVLASS.npy', np.array(df_list_vlass))

## RACSLow3 without DEC +30 and Galactic PLane

In [ ]:
# create a new dataframe with fields around the galactic region and gaalctic cut separations of the catalogue
df_req_fields_dec = df_field_data[df_field_data['DEC_DEG'] < 30]
df_req_fields_gal = df_req_fields_dec[(df_field_data['GAL_LAT'] > 11) | (df_field_data['GAL_LAT'] < -11)]

In [ ]:
plt.figure(figsize=(10, 8))
plt.subplot(111, projection="mollweide")
plt.scatter(np.radians(df_req_fields_gal['GAL_LONG'])-np.pi, np.radians(df_req_fields_gal['GAL_LAT']), c=df_req_fields_gal['SCAN_START'], cmap='jet')
plt.title('Field vs Scan Start Time')
plt.xlabel('Galactic Longitude (in degrees)')
plt.ylabel('Galactic Latitude (in degrees)')
clb = plt.colorbar()
clb.set_label('Scan Start Time (in seconds)')
plt.grid(True)
plt.show()

In [ ]:
# plot the field vs the scan observation time
plt.figure(figsize=(10, 8))
plt.scatter(df_req_fields_gal['RA_DEG'], df_req_fields_gal['DEC_DEG'], c=df_req_fields_gal['SCAN_LEN'], cmap='jet')
plt.title('Field vs Observation Time')
plt.xlabel('RA (in degrees)')
plt.ylabel('DEC (in degrees)')
clb = plt.colorbar()
clb.set_label('Scan Length (in seconds)')
plt.show()

# For full RACSHigh1 coverage

In [ ]:
# save the field names to a numpy array for use in the crossmatch notebook
field_list = df_field_data['FIELD_NAME'].tolist()
# np.save('RACSLow_Fields.npy', field_list)

sbid_list = df_field_data['SBID'].tolist()
# np.save('RACSLow_SBIDs.npy', sbid_list)

cal_sbid_list = df_field_data['CAL_SBID'].tolist()
# np.save('RACSLow_CAL_SBIDs.npy', cal_sbid_list)

df_field_data['UTC_SCAN_START'] = df_field_data['SCAN_START'].apply(mjd2utc)
time_list = df_field_data['UTC_SCAN_START'].tolist()
# np.save('RACSLow_Times.npy', time_list)

# print(field_list)

# Create a new dataframe
df_list = pd.DataFrame()

# Assign values to the columns
df_list['Field Name'] = field_list
df_list['SBID'] = sbid_list
df_list['CAL_SBID'] = cal_sbid_list
df_list['UTC Scan Start Time'] = time_list

df_list.sort_values('UTC Scan Start Time', inplace=True)
df_list.drop_duplicates(subset='Field Name', keep='last', inplace=True)

In [ ]:
sbid = [str(df_list.iloc[i]['SBID']) for i in range(len(df_list))]
cal_sbid = [str(df_list.iloc[i]['CAL_SBID']) for i in range(len(df_list))]

cal_sbid_counts = Counter(cal_sbid)

cal_sbids = list(cal_sbid_counts.keys())
counts = list(cal_sbid_counts.values())

cal_sbid_to_sbid = {}

for i in range(len(df_list)):
    if cal_sbid[i] in cal_sbid_to_sbid:
        if sbid[i] not in cal_sbid_to_sbid[cal_sbid[i]]:
            cal_sbid_to_sbid[cal_sbid[i]].append(sbid[i])
    else:
        cal_sbid_to_sbid[cal_sbid[i]] = [sbid[i]]

# Print the corresponding SBID values for each CAL_SBID value
for cal_sbid_value, sbid_values in cal_sbid_to_sbid.items():
    print(f"CAL_SBID: {cal_sbid_value}, SBID values: {', '.join(sbid_values)}")

sbid_range = [f"{min(cal_sbid_to_sbid[cal_sbids[i]])}-{max(cal_sbid_to_sbid[cal_sbids[i]])}" for i in range(len(cal_sbids))]

plt.figure(figsize=(15, 6))
plt.bar(cal_sbids, counts)
plt.xlabel('SBID Range')
plt.ylabel('Number of Scans')
plt.title('SBID vs Number of Scans')
# Print the text at the bottom of the histogram
for i in range(len(sbid_range)):
    plt.text(i, counts[i], cal_sbids[i], ha='center', va='bottom', fontsize=8, rotation=90, mouseover=cal_sbids[i])

plt.xticks(range(len(sbid_range)), sbid_range, rotation=90)
plt.ylim(top=max(counts)+15)
plt.show()


In [ ]:
# Save the field names to a numpy array for use in the crossmatch notebook
# Convert df_list (1493 fields) to a numpy array
df_list_array = np.array(df_list)

# Save the numpy array to a file
np.save('RACSHigh1_FullList_v1.npy', df_list_array)


## Remove Missing Fields from RACSHigh1 Full Coverage

In [ ]:
import glob

list1 = glob.glob('D:\\ASKAP Astrometry Storage\\RACSHigh_Queries\\RACSHigh_Queries\\selavy-image.i.RACS_*.SB*.cont.RACS_*.beam00.taylor.0.restored.components.xml')

field_names1 = [i.split('image.i.RACS_')[1].split('.SB')[0] for i in list1]
len(field_names1)

In [ ]:
from astropy.table import Table

Table.read(list1[0])

In [ ]:
import numpy as np

list2 = np.load('RACSHigh1_FullList_v1.npy', allow_pickle=True)
field_names2 = [val[0][5:] for val in list2]
len(field_names2)

In [ ]:
unique_in_field_names1 = set(field_names1) - set(field_names2)
print("Unique in field_names1:", unique_in_field_names1)

unique_in_field_names2 = set(field_names2) - set(field_names1)
print("Unique in field_names2:", unique_in_field_names2)

# # Manually adding other problematic field names

# Filter out rows in list2 corresponding to unique_in_field_names2
filtered_list2 = [row for row in list2 if row[0][5:] not in unique_in_field_names2]

# Convert the filtered list back to a numpy array
# filtered_list2 = np.array(filtered_list2)
# np.save('RACSMid1_FullList_v2_rmMissing.npy', filtered_list2)

In [ ]:
# Extract the SBID values from column 2 of filtered_list2
filtered_sbid_values = filtered_list2[:, 1]

# Filter the rows in df_field_data that have the SBID in filtered_sbid_values
filtered_df_field_data = df_field_data[df_field_data['SBID'].isin(filtered_sbid_values)]

# Display the filtered dataframe
print(filtered_df_field_data)

In [ ]:
# plot the field vs the scan observation time
plt.figure(figsize=(10, 8))
plt.scatter(filtered_df_field_data['RA_DEG'], filtered_df_field_data['DEC_DEG'], c=filtered_df_field_data['SCAN_LEN'], cmap='jet')
plt.title('Field vs Observation Time')
plt.xlabel('RA (in degrees)')
plt.ylabel('DEC (in degrees)')
clb = plt.colorbar()
clb.set_label('Scan Length (in seconds)')
plt.show()

## Fields with at least one missing beam

In [ ]:
import glob
from collections import Counter

list1 = glob.glob('D:\\ASKAP Astrometry Storage\\RACSMid_Queries\\RACSMid_Queries\\image.i.RACS_*.SB*.cont.RACS_*.beam**.taylor.0.restored.conv_comp.vot')

field_names3 = [i.split('image.i.RACS_')[1].split('.SB')[0] for i in list1]
print(len(field_names3))
sbid_names3 = [i.split('.SB')[1].split('.cont')[0] for i in list1]
print(len(sbid_names3))

names3_list = list(zip(field_names3, sbid_names3))

field_names3_counts = Counter(field_names3)
print(field_names3_counts)
sbid_names3_counts = Counter(sbid_names3)
print(sbid_names3_counts)

# Find sbid_names whose counts are not equal to 36
field_not_36 = [(field, count) for field, count in field_names3_counts.items() if count%36 != 0]
print(field_not_36)
sbid_not_36 = [(sbid, count) for sbid, count in sbid_names3_counts.items() if count != 36]
print(sbid_not_36)

In [ ]:
1493*36-1451*36

In [ ]:
55327-

In [ ]:
import glob

list1a = glob.glob('D:\\ASKAP Astrometry Storage\\RACSMid_Queries\\RACSMid_Queries\\image.i.RACS_*.SB*.cont.RACS_*.beam00.taylor.0.restored.conv_comp.vot')
list1b = glob.glob('D:\\ASKAP Astrometry Storage\\RACSMid_Queries\\RACSMid_Missing\\image.i.RACS_*.SB*.cont.RACS_*.beam00.taylor.0.restored.conv_comp.vot')
list1 = list1a + list1b
# print(len(list1a), len(list1b), len(list1))

field_names1 = [i.split('image.i.RACS_')[1].split('.SB')[0] for i in list1]
len(field_names1)

In [ ]:
import numpy as np

list2 = np.load('RACSMid1_FullList_v2.npy', allow_pickle=True)
field_names2 = [val[0][5:] for val in list2]
len(field_names2)

In [ ]:
# unique_in_field_names1 = set(field_names1) - set(field_names2)
# print("Unique in field_names1:", unique_in_field_names1)

unique_in_field_names2 = set(field_names2) - set(field_names1)
print("Unique in field_names2:", unique_in_field_names2)

# # Manually adding other problematic field names
# unique_in_field_names2.add('2026+18')
# unique_in_field_names2.add('2054+14')
# unique_in_field_names2.add('0544+46')

# Filter out rows in list2 corresponding to unique_in_field_names2
# filtered_list2 = [row for row in list2 if row[0][5:] not in unique_in_field_names2]

# Convert the filtered list back to a numpy array
# filtered_list2 = np.array(filtered_list2)
# np.save('RACSMid1_FullList_v2_rmMissing.npy', filtered_list2)

In [ ]:
import glob
from collections import Counter

list1a = glob.glob('D:\\ASKAP Astrometry Storage\\RACSMid_Queries\\RACSMid_Queries\\image.i.RACS_*.SB*.cont.RACS_*.beam**.taylor.0.restored.conv_comp.vot')
list1b = glob.glob('D:\\ASKAP Astrometry Storage\\RACSMid_Queries\\RACSMid_Missing\\image.i.RACS_*.SB*.cont.RACS_*.beam**.taylor.0.restored.conv_comp.vot')
list1 = list1a + list1b

# Check for common strings between list1a and list1b
# common_strings = set(list1a).intersection(set(list1b))
# print(f"Number of common strings: {len(common_strings)}")

field_names3 = [i.split('image.i.RACS_')[1].split('.SB')[0] for i in list1]
print(len(field_names3))
sbid_names3 = [i.split('.SB')[1].split('.cont')[0] for i in list1]
print(len(sbid_names3))

names3_list = list(zip(field_names3, sbid_names3))

field_names3_counts = Counter(field_names3)
print(field_names3_counts)
sbid_names3_counts = Counter(sbid_names3)
print(sbid_names3_counts)

# Find sbid_names whose counts are not equal to 36
field_not_36 = [(field, count) for field, count in field_names3_counts.items() if count%36 != 0]
print(field_not_36)
sbid_not_36 = [(sbid, count) for sbid, count in sbid_names3_counts.items() if count != 36]
print(sbid_not_36)

In [ ]:
import os

directory_main = 'D:\ASKAP Astrometry Storage'
directory = os.path.join(directory_main, 'RACSMid_Queries')
directory_racsmid1 = os.path.join(directory, 'RACSMid_Queries') 
directory_racsmid2 = os.path.join(directory, 'RACSMid_Missing')

listz1 = os.path.join(directory_racsmid1, 'image.i.RACS_*.SB*.cont.RACS_*.beam00.taylor.0.restored.conv_comp.vot') 
listz2 = glob.glob(os.path.join(directory_racsmid1, 'image.i.RACS_*.SB*.cont.RACS_*.beam**.taylor.0.restored.conv_comp.vot')) + \
    glob.glob(os.path.join(directory_racsmid2, 'image.i.RACS_*.SB*.cont.RACS_*.beam**.taylor.0.restored.conv_comp.vot'))

In [ ]:
from astropy.table import Table
name = os.path.join(directory_racsmid1, 'image.i.RACS_0000+37.SB20376.cont.RACS_0000+37.beam02.taylor.0.restored.conv_comp.vot')
tab = Table.read(name, format='votable')
tab.columns

In [ ]:
Table(names=['ra', 'dec', 'peak_flux', 'int_flux', 'err_peak_flux', 'err_int_flux', 'maj_axis', 'min_axis', 'pa', 'rms', 'catalog'])
Table(names=['ra', 'err_ra', 'dec', 'err_dec', 'peak_flux', 'err_peak_flux', 'int_flux', 'err_int_flux', 'a', 'err_a', 'b', 'err_b', 'pa', 'err_pa'])

In [ ]:
tab